In [ ]:
!pip install -q \
langchain \
langchain-community \
langchain-huggingface \
langchain-groq \
chromadb \
sentence-transformers \
transformers \
accelerate \
pypdf \
groq

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import Chroma

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline

from transformers import pipeline

from google.colab import files

In [ ]:
def load_documents(pdf_paths):

    docs = []

    for path in pdf_paths:
        loader = PyPDFLoader(path)
        docs.extend(loader.load())

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(docs)

    return chunks

In [ ]:
def create_vectorstore(chunks):

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="./chroma_db"
    )

    return vectorstore

In [ ]:
from langchain_groq import ChatGroq

def build_rag_chain(vectorstore):
    llm = ChatGroq(
        model="llama-3.1-8b-instant",
        api_key="your_groq_api_key_here",
        temperature=0
    )
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 3}
    )
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    def rag_pipeline(question):
        docs = retriever.invoke(question)
        context = format_docs(docs)
        prompt = f"""You are a World Bank development outcomes analyst.
Use the following excerpts from World Bank project documents to answer the question.
Be specific and cite concrete outcomes, numbers, and findings where available.
If the context doesn't contain enough information, say so clearly.

Context:
{context}

Question: {question}

Answer:"""
        response = llm.invoke(prompt)
        return {
            "answer": response.content,
            "sources": [doc.metadata.get("source", "unknown") for doc in docs]
        }
    return rag_pipeline


In [ ]:
uploaded = files.upload()

pdf_paths = list(uploaded.keys())

In [ ]:
chunks = load_documents(pdf_paths)

vectorstore = create_vectorstore(chunks)

chain = build_rag_chain(vectorstore)

In [ ]:
result = chain("What are the project outcomes for Nepal?")

print(result["answer"])

In [ ]:
questions = [
    "What were the key performance indicators and were targets achieved?",
    "What is the overall outcome rating of the project?",
    "What were the achievements of the project development objectives?",
    "What lessons were learned from project implementation?"
]

for q in questions:
    result = chain(q)
    print(f"\nQ: {q}")
    print(f"A: {result['answer']}")
    print(f"Source docs: {result['sources']}")
    print("-" * 50)
